# Orbit Wars — Tüm Node / Graph Metrikleri (Tam Referans)

Bu notebook, **GNN policy** (`pyg_planet_graph`) ve **arrival-credit reward** sisteminde kullanılan **tüm feature ve metrik kanallarını** kaynak kod referanslarıyla birlikte açıklar.

**Motivasyon:** Eğitim replay'lerinde agent'ların kendi gezegenleri arasında filo gönderip geri çağırması (ping-pong) görülebilir. Bu hamle oyunu kazanmaz; ancak **metrik tasarımı** ve **arrival credit** sinyali üzerinden pozitif reward alabilir. Aşağıda her kanalın ne ölçtüğü ve friendly transfer'in hangi kanalları nasıl oynadığı detaylandırılır.

| Katman | Boyut | Kaynak | Reward'da kullanım |
|--------|-------|--------|-------------------|
| Planet obs features | 34 | `obs_builder.py` | GNN girdisi (ham gözlem) |
| Graph node metrics (base) | 10 | `model_graph_pyg._graph_node_metrics` | GNN girdisi + arrival credit |
| Graph node flux (opsiyonel) | +4 | `rl/flux.compute_node_flux_features` | GNN / arrival credit (flag ile) |
| **PyG node input toplam** | **44** veya **48** | `cat(planet_feats, node_metrics)` | Encoder |
| Graph edge metrics | 9 | `model_graph_pyg._graph_edge_metric_matrix` | Sadece GNN edge attr |
| Graph edge flux (opsiyonel) | +2 | `rl/flux` | Sadece GNN (flag ile) |
| Edge bias (attention) | 6 | `obs_builder` | ETA matrisi + SetTransformer bias |
| Fleet features | 11 | `obs_builder._fleet_features` | SetTransformer token (PyG policy'de yok) |
| Global features | 14 | `obs_builder._global_features` | GNN graph summary girdisi |

**Kaynak dosyalar:**
- `obs_builder.py` — ham observation tensörleri
- `model_graph_pyg.py` — graph node/edge metrikleri, PyG encoder
- `rl/held_production.py` — `held_discounted_production` kanalı
- `rl/flux.py` — uçuşta filo akışı (flux) kanalları
- `rl/rewards.py` — `RewardConfig`, `compute_planet_metric_vectors`
- `rl/arrival_credit.py` — counterfactual arrival credit

Notebook `docs/` altındadır; bir sonraki hücre proje kökünü (`Orbit_Wars/`) `sys.path`'e ekler.

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import torch

from obs_builder import (
    PLANET_FEATURE_DIM, FLEET_FEATURE_DIM, GLOBAL_FEATURE_DIM, EDGE_FEATURE_DIM,
    PF_IDX_OWNER_SELF, PF_IDX_SHIPS, PF_IDX_PRODUCTION,
    build_observation,
)
from model_graph_pyg import (
    GRAPH_NODE_METRIC_DIM, GRAPH_NODE_FLUX_DIM, GRAPH_NODE_METRIC_DIM_FLUX,
    GRAPH_EDGE_METRIC_DIM, GRAPH_EDGE_FLUX_DIM,
    PYG_PLANET_FEATURE_DIM, PYG_PLANET_FEATURE_DIM_FLUX,
    PYG_EDGE_FEATURE_DIM, PYG_EDGE_FEATURE_DIM_FLUX,
    INFLUENCE_TAU_TURNS, ETA_NORM,
    _graph_node_metrics, _graph_edge_metric_matrix,
    _estimated_intercept_eta_matrix,
)
from rl.rewards import build_reward_config, compute_planet_metric_vectors, sum_planet_metric_channels
from rl.arrival_credit import build_channel_weights

print("Boyut özeti:")
print(f"  planet_feats           = {PLANET_FEATURE_DIM}")
print(f"  graph node (base)      = {GRAPH_NODE_METRIC_DIM}")
print(f"  graph node (+ flux)    = {GRAPH_NODE_METRIC_DIM_FLUX}")
print(f"  PyG node input         = {PYG_PLANET_FEATURE_DIM} (flux kapalı)")
print(f"  graph edge metrics     = {GRAPH_EDGE_METRIC_DIM}")
print(f"  PyG edge attr          = {PYG_EDGE_FEATURE_DIM} (+ obs edge_bias 6 ayrı)")
print(f"  fleet_feats            = {FLEET_FEATURE_DIM}")
print(f"  global_feats           = {GLOBAL_FEATURE_DIM}")
print(f"  influence tau          = {INFLUENCE_TAU_TURNS} turns")
print(f"  ETA norm cap           = {ETA_NORM} turns")

---

## 1. Planet observation features (34 kanal)

Her gezegen slotu için `obs_builder._planet_features` vektörü. **Normalize edilmiş** değerler; gerçek birim için aşağıdaki "denormalize" sütununa bakın.

| Idx | İsim | Normalizasyon | Gerçek birim | Açıklama |
|-----|------|---------------|--------------|----------|
| 0 | `owner_self` | 0/1 | — | Bu gezegen bana ait mi |
| 1–3 | `owner_enemy_0..2` | 0/1 | — | Hangi düşman slotu (max 3 enemy one-hot) |
| 4 | `owner_neutral` | 0/1 | — | Nötr gezegen |
| 5 | `ships` | `/100` | gemi sayısı | Yüzeydeki mevcut filo |
| 6 | `production` | `/5` | üretim/tur | Gezegen üretim hızı |
| 7 | `radius` | `/5` | birim | Gezegen yarıçapı (launch offset) |
| 8 | `r_sun` | `/50` | birim | Güneşe uzaklık |
| 9 | `cos_theta` | raw | — | Güneş merkezli polar açı (cos) |
| 10 | `sin_theta` | raw | — | Güneş merkezli polar açı (sin) |
| 11 | `is_static` | 0/1 | — | Sabit yörünge |
| 12 | `is_rotating` | 0/1 | — | Dönen yörünge |
| 13 | `is_comet` | 0/1 | — | Komet gezegen grubu |
| 14 | `omega_dt` | raw | rad/tur | Açısal hız (dönen gezegenlerde) |
| 15 | `log_ships` | `log(s+1)/log(1000)` | — | Gemi sayısı log ölçeği |
| 16–19 | `incoming_self_b0..b3` | `/100` | gemi | Bana ait gelen filolar (ETA bucket: 0–5, 5–15, 15–30, 30+) |
| 20–23 | `incoming_enemy_b0..b3` | `/100` | gemi | Düşman gelen filolar (aynı bucket'lar) |
| 24 | `incoming_self_total` | `/100` | gemi | Toplam friendly incoming |
| 25 | `incoming_enemy_total` | `/100` | gemi | Toplam enemy incoming |
| 26 | `incoming_self_min_eta` | `/50` | tur | En yakın friendly varış ETA |
| 27 | `incoming_enemy_min_eta` | `/50` | tur | En yakın düşman varış ETA |
| 28 | `enemy_soonest_ships` | `/100` | gemi | En yakın düşman filosunun boyutu |
| 29 | `enemy_largest_ships` | `/100` | gemi | En büyük düşman incoming filo |
| 30 | `enemy_largest_eta` | `/50` | tur | En büyük düşman filonun ETA'sı |
| 31 | `incoming_net_threat` | `/100` | gemi | `enemy_total - ships - self_total` (self için) |
| 32 | `capture_value` | `/50` | — | Nötr/düşman için: `prod × kalan_tur / max_turns` |
| 33 | `defense_deficit` | `/100` | gemi | Self için: `max(0, enemy_total - ships - prod×enemy_min_eta)` |

**Graph metrikleri bu 34 kanaldan türetilir** — özellikle `ships`, `production`, owner maskeleri, `incoming_*` alanları `_graph_node_metrics` girdisidir.

---

## 2. Fleet features (11 kanal)

Uçuşta her filo için `obs_builder._fleet_features`. PyG planet-graph policy **fleet token kullanmaz**; Set Transformer tabanlı modellerde ayrı token olarak gider.

| Idx | İsim | Normalizasyon | Açıklama |
|-----|------|---------------|----------|
| 0 | `is_self` | 0/1 | Benim filom |
| 1 | `is_enemy` | 0/1 | Düşman filosu |
| 2 | `ships` | `/100` | Filo boyutu |
| 3 | `log_ships` | log scale | Log gemi |
| 4 | `speed_norm` | `/max_speed` | Filo hızı |
| 5 | `cos_angle` | raw | Uçuş yönü cos |
| 6 | `sin_angle` | raw | Uçuş yönü sin |
| 7 | `x_norm` | `x/100 - 0.5` | Konum x |
| 8 | `y_norm` | `y/100 - 0.5` | Konum y |
| 9 | `eta_norm` | `/50` | Tahmini varış ETA (navigation) |
| 10 | `has_pred` | 0/1 | Hedef tahmini var mı |

**Ek routing alanları** (Observation dataclass, flux için):
`fleet_src_slot`, `fleet_tgt_slot`, `fleet_eta_raw`, `fleet_ships_raw`, `fleet_is_self`, `fleet_is_enemy`

---

## 3. Global features (14 kanal)

`obs_builder._global_features` — tüm tahta özeti. PyG encoder'da `graph_summary` ile birleştirilir.

| Idx | İsim | Normalizasyon | Açıklama |
|-----|------|---------------|----------|
| 0 | `turn` | `/200` | Mevcut tur |
| 1 | `omega` | raw | Tahta açısal hızı |
| 2 | `my_planets` | `/25` | Sahip olunan gezegen sayısı |
| 3 | `enemy_planets` | `/25` | Düşman gezegen sayısı |
| 4 | `neutral_planets` | `/25` | Nötr gezegen sayısı |
| 5 | `my_ships` | `/100` | Toplam gemi (yüzey + havada, bana ait) |
| 6 | `enemy_ships` | `/100` | Toplam düşman gemi |
| 7 | `my_prod` | `/50` | Toplam üretim (bana ait gezegenler) |
| 8 | `enemy_prod` | `/50` | Toplam düşman üretim |
| 9 | `num_fleets` | `/25` | Uçuşta filo sayısı |
| 10 | `incoming_other` | `/100` | Bana gelen toplam düşman gemi (tüm gezegenler) |
| 11 | `incoming_self` | `/100` | Bana gelen toplam friendly gemi |
| 12 | `my_capture_value` | `/5000` | `my_prod × kalan_tur / max_turns` |
| 13 | `air_advantage` | `/100` | `(my_air - enemy_air)` havada gemi farkı |

---

## 4. Edge bias (6 kanal) — attention / ETA yardımcısı

Token sırası: `[planets (P_max), fleets (F_max), global (1)]`. Her token çifti `(i,j)` için `edge_bias[i,j,:]`:

| Idx | İsim | Formül | Kullanım |
|-----|------|--------|----------|
| 0 | `distance` | `‖pos_i - pos_j‖ / 100` | Mesafe; ETA fallback |
| 1 | `dx` | `(x_j - x_i) / 100` | Yönsel fark |
| 2 | `dy` | `(y_j - y_i) / 100` | Yönsel fark |
| 3 | `sin_dphi` | `sin(φ_j - φ_i)` | Güneş etrafında açı farkı (yörünge ilişkisi) |
| 4 | `cos_dphi` | `cos(φ_j - φ_i)` | Açı farkı cos |
| 5 | `crosses_sun` | 0/1 | Segment güneş yarıçapını kesiyor mu |

Graph metriklerinde **planet–planet** alt-blok (`edge_bias[:n, :n]`) kullanılır. `_estimated_intercept_eta_matrix` dönen gezegenler için intercept ETA hesaplar; fallback olarak `distance/speed` kullanır.

---

## 5. Graph node metrics — base 10 kanal

Kaynak: `model_graph_pyg._graph_node_metrics`. Her gezegen `g` için **hedef-merkezli** (g'ye bakarak) skalarlar.

### Ortak tanımlar

```text
ships_i        = planet_feats[i, ships] × 100
prod_i         = planet_feats[i, production] × 5
is_self_i      = owner_self one-hot
is_enemy_i     = sum(owner_enemy slots)
eta(i→j)       = intercept ETA (tur), max ~50
K_ij           = exp(-eta(i→j) / τ)     τ = INFLUENCE_TAU_TURNS = 12
K^sup_ij       = K_ij, diagonal = 0       (kaynak ≠ hedef)
incoming_friendly_g = planet_feats[g, incoming_self_total] × 100
incoming_enemy_g    = planet_feats[g, incoming_enemy_total] × 100
```

### Kanal tablosu

| # | Kod adı | Formül | Ölçek | Aralık (tipik) | Anlam |
|---|---------|--------|-------|----------------|-------|
| **0** | `friendly_influence` | `Σ_i ships_i · is_self_i · K_{i→g}` | `/100` | 0–10+ | Bana ait gemilerin g'ye **uzaklık-ağırlıklı** etkisi (tüm tahta) |
| **1** | `enemy_influence` | `Σ_i ships_i · is_enemy_i · K_{i→g}` | `/100` | 0–10+ | Düşman gemilerin g'ye etkisi |
| **2** | `influence_balance` | `friendly_influence - enemy_influence` | — | −10..+10 | Net baskınlık (doğrusal kombinasyon) |
| **3** | `friendly_support` | `Σ_{i≠g} ships_i · is_self_i · K_{i→g}` | `/100` | 0–10+ | **g dışındaki** friendly gemilerin g'ye desteği (diagonal sıfır) |
| **4** | `supportability` | `friendly_support / (enemy_influence + incoming_enemy + 1)` | clamp/10 | 0–1 | g ne kadar "desteklenebilir" (payda tehdit) |
| **5** | `vulnerability` | `enemy_influence / (ships_g + incoming_friendly + 1)` | clamp/10 | 0–1 | g ne kadar savunmasız (pay: garrison + friendly incoming) |
| **6** | `production_potential` | `Σ_i prod_i · K_{i→g}` | `/25` | 0–5+ | Tüm gezegenlerin üretiminin g'ye diffusion'ı |
| **7** | `friendly_production_potential` | `Σ_i prod_i · is_self_i · K_{i→g}` | `/25` | 0–5+ | Sadece **bana ait** gezegen üretiminin g'ye diffusion'ı |
| **8** | `strategic_potential` | `production_scaled + friendly_scaled - enemy_scaled` | — | — | Özet stratejik skor (doğrusal komb.) |
| **9** | `held_discounted_production` | aşağıda | `/500` | 0–2+ | **Sadece self-owned** g'de: gelecekte tutulabilecek üretim değeri |

### Kanal 9 — `held_discounted_production` (detay)

Kaynak: `rl/held_production.py`

```text
R = max(0, max_turns - turn)          # kalan tur
v_g = vulnerability[g]              # [0, 1]
x_g = survival_min + (1 - v_g) × (survival_max - survival_min)
      varsayılan: survival_min=0.5, survival_max=0.99
geom_g = 1 + x_g + x_g² + ... + x_g^{R-1} = (1 - x_g^R) / (1 - x_g)

held_discounted_production[g] = prod_g × is_self_g × geom_g / held_production_norm
held_production_norm = 500 (varsayılan)
```

**Kritik:** Düşük `vulnerability` → yüksek `x_g` → yüksek `geom` → yüksek held production. Gemi **hedef gezegene** varınca garrison artar → vulnerability düşer → bu kanal **sıçrayabilir**.

---

## 6. Graph node flux — opsiyonel +4 kanal

`RewardConfig.enable_flux_metrics=True` iken `_graph_node_metrics`'e eklenir. Kaynak: `rl/flux.compute_node_flux_features`.

Uçuşta filolar için yönlü akış matrisi `M[i,j]` = i'den j'ye ETA-ağırlıklı gemi akışı:

| Ağırlık modu | Formül |
|--------------|--------|
| `raw` | `ships` |
| `linear` | `ships × max(0, 1 - eta/eta_norm)` |
| `exp_decay` | `ships × exp(-eta/tau)` |

| # | Kod adı | Formül | Ölçek |
|---|---------|--------|-------|
| **10** | `flux_friendly_in` | `Σ_i M_friendly[i, g]` | `/ship_norm` (50) |
| **11** | `flux_friendly_out` | `Σ_j M_friendly[g, j]` | `/ship_norm` |
| **12** | `flux_enemy_in` | `Σ_i M_enemy[i, g]` | `/ship_norm` |
| **13** | `flux_enemy_out` | `Σ_j M_enemy[g, j]` | `/ship_norm` |

**Friendly transfer sırasında:** Launch anında `flux_friendly_out` kaynakta artar, varışta `flux_friendly_in` hedefte artar; arrival credit flux ağırlıkları açıksa bu da sinyal üretir.

---

## 7. Graph edge metrics — 9 kanal (GNN only)

Kaynak: `model_graph_pyg._graph_edge_metric_matrix`. Her yönlü planet çifti `(i → j)` için. **Arrival credit'te kullanılmaz** — sadece PyG `edge_attr`.

| # | Kod adı | Formül | Açıklama |
|---|---------|--------|----------|
| 0 | `eta_norm` | `clamp(eta(i→j), 0, 50) / 50` | Normalize intercept ETA |
| 1 | `movement_resistance` | `(eta_norm × (1 + crosses_sun)) / 2` | Hareket zorluğu (güneş geçişi cezalı) |
| 2 | `same_owner` | 1 iff i,j aynı owner (self/enemy/neutral) | Aynı takım mı |
| 3 | `src_is_self` | `is_self_i` | Kaynak bana ait mi |
| 4 | `tgt_is_self` | `is_self_j` | Hedef bana ait mi |
| 5 | `tgt_is_enemy` | `is_enemy_j` | Hedef düşman mı |
| 6 | `tgt_is_neutral` | `is_neutral_j` | Hedef nötr mü |
| 7 | `strategic_gradient` | `clamp(strategic_j - strategic_i, -5, 5) / 5` | Stratejik potansiyel farkı |
| 8 | `flow_opportunity` | `relu(strategic_j - strategic_i) / (eta + 1)` | i→j yönünde "fırsat" (tek yönlü) |

**PyG edge_attr toplam:** `EDGE_FEATURE_DIM (6)` obs mesafe + `GRAPH_EDGE_METRIC_DIM (9)` = **15** (flux kapalı).

Flux açıkken +2 kanal: `friendly_flux_ij`, `enemy_flux_ij` (`rl/flux.compute_pair_flux_edge_features`) → toplam **17**.

---

## 8. PyG encoder birleşimi

```text
node_x[g] = concat(planet_feats[g], graph_node_metrics[g])   # (44,) flux kapalı
edge_attr(e: i→j) = concat(edge_bias[i,j,:6], graph_edge_metrics[i,j,:9])
global → graph pooling + global_feats → policy heads
```

Policy **metrikleri doğrudan optimize etmez** — observation olarak görür. Reward ise arrival credit üzerinden aynı node metriklerinin **counterfactual delta**sını kullanır.

---

## 9. Arrival credit — reward kanal ağırlıkları

`graph_native` preset (`rl/rewards.build_reward_config`) varsayılanları:

| Kanal | Ağırlık | Yorum |
|-------|---------|-------|
| friendly_influence | **+1.0** | Artış iyi |
| enemy_influence | **−1.0** | Artış kötü |
| influence_balance | 0.0 | (redundant) |
| friendly_support | **+1.0** | Artış iyi |
| supportability | **+1.0** | Artış iyi |
| vulnerability | **−1.0** | Artış kötü (negatif ağırlık) |
| production_potential | **+1.0** | Artış iyi |
| friendly_production_potential | **+1.0** | Artış iyi |
| strategic_potential | 0.0 | (redundant) |
| held_discounted_production | **+1.0** | Artış iyi |
| flux_* (4 kanal) | 0.0 | flux kapalıyken |

**Credit formülü** (`rl/arrival_credit.py`):

```text
credit = w(eta_launch) × arrival_credit_weight × reward_scale × Σ_c w_c · Δmetric_c

Counterfactual modda:
  Δ = metric(actual_state) - metric(counterfactual_state)
  CF: filo hedefe varmamış + (opsiyonel) kaynağa geri dönmüş
  scope=full_graph: tüm gezegenlerde toplam Δ (owner_filter=self ise sadece self-owned)
```

`w(eta) = exp(-eta/tau)` — yakın varışlar daha yüksek credit.

---

## 10. Kendi gezegenleri arası transfer — metrikleri nasıl oynar?

Senaryo: Gezegen **A** (kaynak) → **B** (hedef), ikisi de **self-owned**, `S` gemi gönderilir.

### Anında (launch)
- A: `ships` ↓, `flux_friendly_out` ↑ (flux açıksa)
- B: `incoming_self_*` ↑ (ETA boyunca)
- **Toplam empire gemi sabit** (havada)

### Varış anı — counterfactual Δ = metric(actual) − metric(filo_varmasaydı)

**Launch kaynak gezegenin birçok kanalını düşürür.** CF'de gemi kaynakta kalır; actual'da kaynak zayıflar:

| Kanal (kaynak A) | Actual vs CF | w·Δ (demo) |
|------------------|--------------|------------|
| `friendly_influence` | **↓** | **−0.18** |
| `held_discounted_production` | **↓** (garrison↓) | **−0.06** |
| `vulnerability` | **↑** | −0.01 |
| `friendly_support` | **↑** (hedef B güçlenince kernel desteği) | **+0.62** |
| `supportability` | **↑** (aynı diffusion) | **+0.33** |

Hedefte de her kanal iyileşmez:

| Kanal (hedef B) | Actual vs CF | w·Δ (demo) |
|-----------------|--------------|------------|
| `held_discounted_production` | **↑** | **+0.21** |
| `vulnerability` | **↓** | +0.08 |
| `friendly_support` | **↓** (kaynak zayıfladı) | **−0.62** |
| `supportability` | **↓** | **−0.39** |

### Scope'a göre net credit (A=200 gemi, B=15 gemi, 80 transfer)

| Scope | A slot | B slot | Net |
|-------|--------|--------|-----|
| `target_source` | +0.71 | −0.55 | **+0.16** |
| `full_graph` + `owner_filter=self` | — | — | **≈ 0** |
| Round-trip A→B→A | — | — | **0** |

`run_held_cf_v2` **`target_source`** kullanıyor → kaynak ve hedef **ayrı toplanır**. Kaynakta düşen kanallar varken diffusion (`friendly_support`) kaynak slotunu pozitife çevirebilir; hedefte `held_prod` kazancı ile birleşince bacak net **+** olabilir.

### Neden ping-pong yine görülür?

1. **`target_source` double-counting:** Kaynak diffusion kazancı + hedef held_prod kazancı, hedef supportability kaybını aşabilir.
2. **Asimetrik çiftler:** Düşük garrison / yüksek prod hedefe transfer → büyük `held_prod` sıçraması; geri dönüşte tersi.
3. **Paralel filolar:** Round-trip tek başına 0 olsa bile eşzamanlı bacaklar pozitif toplam verir.
4. **Scripted bot aynı metrikleri optimize eder.**

**Sonuç:** Ping-pong oyunu kazanmaz ama **metrik uzayında lokal iyileştirme** yapabilir; PPO bunu öğrenirse replay'de gördüğün davranış ortaya çıkar.

**Olası düzeltmeler (referans):**
- Friendly→friendly transfer'e arrival credit verme veya `same_owner` edge feature ile gate
- Empire-toplam metrik kullan (diffusion conservation)
- `held_discounted_production`'da garrison transferini nötr say
- `terminal_scale` ile oyun sonucunu daha baskın yap

---

## 11. Canlı demo — friendly transfer metrik delta

Aşağıdaki hücre toy bir observation üzerinde A→B transfer simüle eder ve **tüm 10 base kanal** + weighted arrival credit skorunu yazdırır.

In [ ]:
CHANNEL_NAMES = [
    "friendly_influence", "enemy_influence", "influence_balance",
    "friendly_support", "supportability", "vulnerability",
    "production_potential", "friendly_production_potential",
    "strategic_potential", "held_discounted_production",
    "flux_friendly_in", "flux_friendly_out", "flux_enemy_in", "flux_enemy_out",
]

def toy_env():
    return {
        "planets": [
            [0, 0, 70.0, 70.0, 3.0, 200, 4],   # A: self, prod=4, many ships
            [1, 0, 80.0, 70.0, 2.0, 15, 3],    # B: self, prod=3, few ships (vulnerable)
            [2, 1, 30.0, 30.0, 2.0, 80, 2],    # enemy
            [3, -1, 20.0, 80.0, 2.0, 10, 1],   # neutral
        ],
        "fleets": [],
        "initial_planets": [
            [0, 0, 70.0, 70.0, 3.0, 200, 4],
            [1, 0, 80.0, 70.0, 2.0, 15, 3],
            [2, 1, 30.0, 30.0, 2.0, 80, 2],
            [3, -1, 20.0, 80.0, 2.0, 10, 1],
        ],
        "comet_planet_ids": [],
        "comets": [],
    }

def simulate_transfer(obs, src_slot=0, tgt_slot=1, ships=50):
    """Launch: remove ships from src. Arrival: add to tgt (merge)."""
    import copy
    from dataclasses import replace
    before = copy.deepcopy(obs)
    after = copy.deepcopy(obs)
    # launch state (simplified: instant arrival for metric demo)
    s = float(obs.planet_feats[src_slot, PF_IDX_SHIPS] * 100)
    t = float(obs.planet_feats[tgt_slot, PF_IDX_SHIPS] * 100)
    send = min(ships, s - 1)
    after.planet_feats[src_slot, PF_IDX_SHIPS] = (s - send) / 100.0
    after.planet_feats[tgt_slot, PF_IDX_SHIPS] = (t + send) / 100.0
    return before, after, send

cfg = build_reward_config("graph_native")
weights = build_channel_weights(cfg)

obs = build_observation(toy_env(), player=0, angular_velocity=0.0, p_max=8, f_max=4, turn=100, max_turns=250)
before, after, sent = simulate_transfer(obs, src_slot=0, tgt_slot=1, ships=80)

m0 = compute_planet_metric_vectors(before, cfg=cfg, turn=100)
m1 = compute_planet_metric_vectors(after, cfg=cfg, turn=100)
delta = m1 - m0

print(f"Simüle transfer: slot 0 → slot 1, {sent:.0f} gemi\n")
print(f"{'Kanal':<32} {'A Δ':>8} {'B Δ':>8} {'Toplam':>8} {'w':>6} {'w·Δ':>8}")
print("-" * 72)
for c, name in enumerate(CHANNEL_NAMES[:10]):
    dA, dB = float(delta[0, c]), float(delta[1, c])
    tot = dA + dB
    w = float(weights[c])
    print(f"{c:2d} {name:<29} {dA:8.4f} {dB:8.4f} {tot:8.4f} {w:6.1f} {w*tot:8.4f}")

weighted_total = float(np.dot(weights[:10], delta[:2].sum(axis=0)))
print("-" * 72)
print(f"Empire-toplam weighted score (A+B only): {weighted_total:+.4f}")
print(f"(Gerçek arrival credit full_graph scope tüm self gezegenleri toplar)")

full_delta = delta.sum(axis=0)
full_weighted = float(np.dot(weights[:10], full_delta))
print(f"4 gezegen toplam weighted score: {full_weighted:+.4f}")

---

## 12. Tüm kanalları tek tabloda (14 + obs referans)

Arrival credit ve scripted graph bot **aynı 10 base kanalı** kullanır (flux kapalı koşuda).

In [ ]:
cfg = build_reward_config("graph_native")
w = build_channel_weights(cfg)

rows = []
for i, name in enumerate(CHANNEL_NAMES):
    layer = "node_base" if i < 10 else "node_flux"
    rows.append({
        "idx": i,
        "name": name,
        "layer": layer,
        "arrival_credit_weight": float(w[i]),
        "in_pyg_encoder": i < 10 or cfg.enable_flux_metrics,
        "in_arrival_credit": float(w[i]) != 0.0,
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows:
        print(r)

---

## 13. Hızlı referans — sabitler

| Sabit | Değer | Dosya |
|-------|-------|-------|
| `INFLUENCE_TAU_TURNS` | 12 | `model_graph_pyg.py` |
| `ETA_NORM` | 50 | `model_graph_pyg.py` |
| `held_production_norm` | 500 | `RewardConfig` |
| `hold_survival_min/max` | 0.5 / 0.99 | `RewardConfig` |
| `GRAPH_NODE_METRIC_DIM` | 10 | `model_graph_pyg.py` |
| `GRAPH_NODE_FLUX_DIM` | 4 | `model_graph_pyg.py` / `rl/flux.py` |
| `GRAPH_EDGE_METRIC_DIM` | 9 | `model_graph_pyg.py` |
| `GRAPH_EDGE_FLUX_DIM` | 2 | `model_graph_pyg.py` |
| `PLANET_FEATURE_DIM` | 34 | `obs_builder.py` |
| `PYG_PLANET_FEATURE_DIM` | 44 | 34 + 10 |

---

**İlgili notebook'lar:** `pyg_graph_pipeline_walkthrough.ipynb`, `orbit_wars_flux_walkthrough.ipynb`, `reward_system_explained.ipynb`, `observation_system_explained.ipynb`